In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchcrf import CRF
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score
import numpy as np
import fasttext
import warnings

warnings.filterwarnings("ignore")

# Конфигурация

In [2]:
DATA_FILE = "./allergens.conll"
FASTTEXT_BIN_PATH = "./fasttext.bin" 
EMBEDDING_DIM = 300  
HIDDEN_DIM = 128
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 0.001
DROPOUT = 0.5
FREEZE_EMBEDDINGS = False 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 86
NUM_LAYERS = 1

TRAIN_RATIO = 0.8
VAL_RATIO = 0.10
TEST_RATIO = 0.10

torch.manual_seed(SEED)
np.random.seed(SEED)


# Загрузка данных в формате CoNLL

In [3]:
def read_conll(filepath):
    sentences = []
    tags = []
    current_sent = []
    current_tags = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sent:
                    sentences.append(current_sent)
                    tags.append(current_tags)
                    current_sent = []
                    current_tags = []
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            word = parts[0]
            tag = parts[-1]
            current_sent.append(word)
            current_tags.append(tag)
    if current_sent:
        sentences.append(current_sent)
        tags.append(current_tags)
    return sentences, tags


print("Loading CoNLL data...")
sentences, tags = read_conll(DATA_FILE)
print(f"Loaded {len(sentences)} sentences.")

Loading CoNLL data...
Loaded 6243 sentences.


# Разбиение на обучающую, валидационную и тестовую выборки

In [4]:
train_val_sentences, test_sentences, train_val_tags, test_tags = train_test_split(
    sentences, tags, test_size=TEST_RATIO, random_state=SEED, shuffle=True
)
train_sentences, val_sentences, train_tags, val_tags = train_test_split(
    train_val_sentences,
    train_val_tags,
    test_size=VAL_RATIO / (TRAIN_RATIO + VAL_RATIO),
    random_state=SEED,
    shuffle=True,
)

print(f"Train: {len(train_sentences)} sentences")
print(f"Validation: {len(val_sentences)} sentences")
print(f"Test: {len(test_sentences)} sentences")

Train: 4993 sentences
Validation: 625 sentences
Test: 625 sentences


# Загрузка модели FastText (.bin) и построение матрицы векторных представлений

In [5]:
print(f"Loading FastText model from {FASTTEXT_BIN_PATH} ...")
ft = fasttext.load_model(FASTTEXT_BIN_PATH)
print("FastText model loaded.")


def build_vocab_and_embedding_matrix(
    train_sentences, train_tags, ft_model, embedding_dim, min_freq=1):
    word_counts = {}
    for sent in train_sentences:
        for w in sent:
            word_counts[w] = word_counts.get(w, 0) + 1

    vocab_words = ["<PAD>", "<UNK>"] + [
        w
        for w, c in sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        if c >= min_freq
    ]
    word2idx = {w: i for i, w in enumerate(vocab_words)}

    embedding_matrix = torch.zeros(len(vocab_words), embedding_dim)
    for word, idx in word2idx.items():
        if word == "<PAD>":
            embedding_matrix[idx] = torch.zeros(embedding_dim)
        else:
            vec = ft_model.get_word_vector(word)
            embedding_matrix[idx] = torch.tensor(vec, dtype=torch.float32)

    # Build tag vocabulary
    unique_tags = set()
    for tag_seq in train_tags:
        unique_tags.update(tag_seq)
    tag2idx = {t: i for i, t in enumerate(sorted(unique_tags))}

    return word2idx, tag2idx, embedding_matrix

word2idx, tag2idx, embedding_matrix = build_vocab_and_embedding_matrix(
    train_sentences, train_tags, ft, EMBEDDING_DIM, min_freq=1
)
idx2tag = {i: t for t, i in tag2idx.items()}
vocab_size = len(word2idx)
tagset_size = len(tag2idx)
print(f"Vocabulary size: {vocab_size}, Tagset size: {tagset_size}")


Loading FastText model from ./fasttext.bin ...
FastText model loaded.
Vocabulary size: 8307, Tagset size: 25


# Датасет и DataLoader

In [6]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, word2idx, tag2idx):
        self.sentences = sentences
        self.tags = tags
        self.word2idx = word2idx
        self.tag2idx = tag2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags = self.tags[idx]
        word_ids = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in words]
        tag_ids = [self.tag2idx[t] for t in tags]
        return torch.tensor(word_ids, dtype=torch.long), torch.tensor(
            tag_ids, dtype=torch.long
        )


def collate_fn(batch):
    word_ids, tag_ids = zip(*batch)
    word_ids_padded = pad_sequence(
        word_ids, batch_first=True, padding_value=word2idx["<PAD>"]
    )
    tag_ids_padded = pad_sequence(tag_ids, batch_first=True, padding_value=-1)
    mask = (word_ids_padded != word2idx["<PAD>"]).float()
    return word_ids_padded, tag_ids_padded, mask


train_dataset = NERDataset(train_sentences, train_tags, word2idx, tag2idx)
val_dataset = NERDataset(val_sentences, val_tags, word2idx, tag2idx)
test_dataset = NERDataset(test_sentences, test_tags, word2idx, tag2idx)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)


# Модель BiLSTM+CRF

In [7]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        tagset_size,
        embedding_dim,
        hidden_dim,
        dropout,
        embedding_matrix=None,
        freeze_embeddings=False,
    ):
        super(BiLSTMCRF, self).__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=word2idx["<PAD>"]
        )
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
            if freeze_embeddings:
                self.embedding.weight.requires_grad = False

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            num_layers=NUM_LAYERS,
            bidirectional=True,
            batch_first=True,
            dropout=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, x, mask):
        embeddings = self.embedding(x)
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = self.dropout(lstm_out)
        emissions = self.hidden2tag(lstm_out)
        return emissions

    def loss(self, x, tags, mask):
        emissions = self.forward(x, mask)
        return -self.crf(emissions, tags, mask=mask.bool(), reduction="mean")

    def predict(self, x, mask):
        emissions = self.forward(x, mask)
        return self.crf.decode(emissions, mask=mask.bool())


model = BiLSTMCRF(
    vocab_size=vocab_size,
    tagset_size=tagset_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    embedding_matrix=embedding_matrix,
    freeze_embeddings=FREEZE_EMBEDDINGS,
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# Модель BiLSTM+CRF

In [8]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for x, y, mask in loader:
        x, y, mask = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        loss = model.loss(x, y, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, idx2tag):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for x, y, mask in loader:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            preds = model.predict(x, mask)
            for i in range(len(preds)):
                seq_len = mask[i].sum().int().item()
                pred_tags = [idx2tag[p] for p in preds[i][:seq_len]]
                true_tags = [idx2tag[t.item()] for t in y[i][:seq_len]]
                all_preds.append(pred_tags)
                all_labels.append(true_tags)
    f1 = f1_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, digits=4)
    return f1, report

# Цикл обучения с валидацией

In [9]:
print("Starting training...")
best_val_f1 = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_f1, _ = evaluate(model, val_loader, idx2tag)
    print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_bilstm_crf_fasttext.pt")
        print(f"  -> Saved best model (Val F1={val_f1:.4f})")

print(f"\nBest validation F1: {best_val_f1:.4f}")

Starting training...
Epoch  1 | Train Loss: 62.3183 | Val F1: 0.6643
  -> Saved best model (Val F1=0.6643)
Epoch  2 | Train Loss: 17.1033 | Val F1: 0.7840
  -> Saved best model (Val F1=0.7840)
Epoch  3 | Train Loss: 12.1256 | Val F1: 0.8228
  -> Saved best model (Val F1=0.8228)
Epoch  4 | Train Loss: 10.3078 | Val F1: 0.8296
  -> Saved best model (Val F1=0.8296)
Epoch  5 | Train Loss: 9.2193 | Val F1: 0.8406
  -> Saved best model (Val F1=0.8406)
Epoch  6 | Train Loss: 8.5010 | Val F1: 0.8415
  -> Saved best model (Val F1=0.8415)
Epoch  7 | Train Loss: 7.9686 | Val F1: 0.8421
  -> Saved best model (Val F1=0.8421)
Epoch  8 | Train Loss: 7.6240 | Val F1: 0.8443
  -> Saved best model (Val F1=0.8443)
Epoch  9 | Train Loss: 7.2136 | Val F1: 0.8449
  -> Saved best model (Val F1=0.8449)
Epoch 10 | Train Loss: 6.8652 | Val F1: 0.8461
  -> Saved best model (Val F1=0.8461)
Epoch 11 | Train Loss: 6.6973 | Val F1: 0.8486
  -> Saved best model (Val F1=0.8486)
Epoch 12 | Train Loss: 6.4683 | Val F1: 

# Финальная оценка на тестовой выборке

In [10]:
print("\n--- Loading best model and evaluating on TEST set ---")
model.load_state_dict(torch.load("best_bilstm_crf_fasttext.pt"))
test_f1, test_report = evaluate(model, test_loader, idx2tag)
print(f"Test F1: {test_f1:.4f}")
print("\nDetailed Test Report:")
print(test_report)


--- Loading best model and evaluating on TEST set ---
Test F1: 0.8346

Detailed Test Report:
              precision    recall  f1-score   support

         EGG     0.8643    0.8721    0.8682       219
        FISH     0.7419    0.6866    0.7132       134
      GLUTEN     0.7819    0.7471    0.7641       427
     LACTOSE     0.7882    0.7801    0.7841       973
        MEAT     0.7985    0.7562    0.7768       283
      PEANUT     0.8286    0.8286    0.8286        35
      SESAME     0.8605    0.8605    0.8605        43
   SHELLFISH     0.6792    0.6923    0.6857        52
      SODIUM     0.9220    0.9347    0.9283       949
         SOY     0.7462    0.8220    0.7823       236
       SUGAR     0.9126    0.9381    0.9252       824
    TREE_NUT     0.5731    0.5904    0.5816       166

   micro avg     0.8333    0.8360    0.8346      4341
   macro avg     0.7914    0.7924    0.7915      4341
weighted avg     0.8328    0.8360    0.8341      4341



# Сохранение полной модели для последующего использования

In [11]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "word2idx": word2idx,
        "idx2tag": idx2tag,
        "tag2idx": tag2idx,
        "embedding_matrix": embedding_matrix,
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM,
            "dropout": DROPOUT,
        },
    },
    "full_model_fasttext.pt",
)
print("\nFull model saved to full_model_fasttext.pt")


Full model saved to full_model_fasttext.pt
